# Exploration de l'API TED (marchés publics européens)

**Projet :** Plateforme d'automatisation de la prospection (veille AO / RFP / CFP)
**Module :** 1 - Collecte
**Source :** TED (Tenders Electronic Daily), le journal officiel des marchés publics de l'UE.

## Pourquoi TED

La BCEAO publie très peu de cybersécurité. Je cherche une source avec du volume et de vraies
opportunités cyber. TED est une **API officielle** : données structurées (pas de HTML à scraper),
énorme volume (toute l'Europe), filtrage par mots-clés / pays / catégorie.

## Ma démarche

Je ne connais pas cette API. Je la découvre **par tâtonnements** : je tente un appel, je lis ce que
l'API répond (surtout ses erreurs, très instructives), et j'ajuste. Je veux qu'aucun nom de champ
n'apparaisse "par magie" : chaque nom que j'utilise, je montre d'abord d'où il vient.

## 1. Est-ce que l'API répond ?

Je fais l'appel le plus simple possible et je regarde le **code HTTP** (200 = OK, 4xx = j'ai fait
une erreur, 5xx = souci serveur).

In [1]:
import requests
import json
from collections import Counter
from datetime import date

URL = "https://api.ted.europa.eu/v3/notices/search"

r = requests.post(URL, json={"query": "publication-date>=today(-1)"})
print("Code HTTP :", r.status_code)

Code HTTP : 400


**400** : l'API n'est pas contente. Ce n'est pas un échec, c'est une piste : elle va me dire
ce qui manque. Je lis le message.

In [2]:
print(json.dumps(r.json(), indent=2, ensure_ascii=False))

{
  "message": "Validation error",
  "error": [
    {
      "objectName": "publicExpertSearchRequestV1",
      "field": "fields",
      "message": "must not be empty"
    }
  ]
}


Le message dit : le champ **`fields` ne doit pas être vide**. L'API veut que je précise quelles
informations je veux pour chaque annonce. Mais je ne connais aucun nom de champ valide... Je vais
en tenter un au hasard pour voir comment l'API réagit.

## 2. Découvrir les noms de champs (grâce à l'erreur)

Je tente un nom intuitif au hasard : `"title"`.

In [3]:
r = requests.post(URL, json={"query": "publication-date>=today(-1)", "fields": ["title"]})
print("Code HTTP :", r.status_code)
print(json.dumps(r.json(), ensure_ascii=False)[:350])

Code HTTP : 400
{"message": "Parameter 'fields' contains unsupported value (supported values are: sme-part,touchpoint-gateway-ted-esen,submission-url-lot,organisation-person-addinfo-part,no-negocaition-necessary-lot,BT-13(t)-Part,organisation-city-serv-prov,result-framework-maximum-value-cur-notice,BT-821-Lot,touchpoint-partname-tenderer,touchpoint-internet-addres


Encore 400, mais le message est en or : « unsupported value (**supported values are:** ... ) »
suivi d'une longue liste. En me trompant, l'API m'a **listé tous les champs valides**. Je récupère
cette liste et je la compte.

In [4]:
message = r.json()["message"]
liste_brute = message.split("supported values are:")[1].rstrip(")").strip()
champs = [c.strip() for c in liste_brute.split(",") if c.strip()]
print("Nombre de champs proposés par l'API :", len(champs))
print("Exemples (10 premiers) :", champs[:5], "...")

Nombre de champs proposés par l'API : 1830
Exemples (10 premiers) : ['sme-part', 'touchpoint-gateway-ted-esen', 'submission-url-lot', 'organisation-person-addinfo-part', 'no-negocaition-necessary-lot'] ...


**Attention :** ces 1830, ce sont des **champs** (les colonnes possibles : titre, date, pays...),
PAS le nombre d'annonces. Je verrai le nombre d'annonces plus loin. Pour l'instant, je cherche dans
cette liste les champs dont j'ai besoin.

## 3. Chercher mes champs dans la liste (avant de les utiliser)

Je ne veux pas utiliser un nom de champ sans l'avoir vu dans la liste. Donc je fouille la liste avec
des mots-clés correspondant aux informations utiles : identifiant, titre, date, pays, lien, catégorie.
J'écarte les champs préfixés `BT-` (codes techniques peu lisibles) juste pour la lisibilité.

In [5]:
for mot in ["publication", "number", "title", "deadline", "buyer-country", "buyer-name", "links", "cpv"]:
    trouves = [c for c in champs if mot in c.lower() and not c.startswith("BT-")][:5]
    print(f"{mot:16} -> {trouves}")

publication      -> ['publication-date', 'publication-number', 'publication-name', 'notice-preferred-publication-date']
number           -> ['award-criterion-number-fixed-glo', 'framework-maximum-participants-number-lot', 'number-tender-applications-ipi-measure-res', 'OPA-36-Part-Number', 'award-criterion-number-fixed-lot']
title            -> ['review-title', 'announcement-title', 'contract-title', 'notice-title', 'title-part']
deadline         -> ['deadline-receipt-tender-date-lot', 'tender-validity-deadline-unit-lot', 'deadline-time-part', 'deadline-receipt-expressions-time-lot', 'deadline-receipt-request']
buyer-country    -> ['buyer-country-sub', 'buyer-country']
buyer-name       -> ['buyer-name']
links            -> ['links']
cpv              -> ['classification-cpv']


Maintenant je vois d'où viennent les noms que je vais utiliser. En particulier,
**`publication-number`** apparaît bien dans la liste (sous "publication" et "number") : c'est
l'identifiant de l'annonce. Je retiens : `publication-number`, `notice-title`, `buyer-name`,
`buyer-country`, `publication-date`, `deadline`, `classification-cpv`, `links`.

## 4. Premier appel qui marche, et surtout : que renvoie l'API ?

J'utilise `publication-number` (trouvé dans la liste) pour faire un appel valide. Puis, au lieu de
supposer la forme de la réponse, je **regarde les clés** que l'API me renvoie.

In [6]:
r = requests.post(URL, json={"query": "publication-date>=today(-1)",
                              "fields": ["publication-number"], "limit": 3})
reponse = r.json()
print("Code HTTP :", r.status_code)
print("\nLes CLÉS que l'API me renvoie :")
for cle, val in reponse.items():
    apercu = f"liste de {len(val)} éléments" if isinstance(val, list) else val
    print(f"   {cle} = {apercu}")

Code HTTP : 200

Les CLÉS que l'API me renvoie :
   notices = liste de 3 éléments
   totalNoticeCount = 7321
   iterationNextToken = None
   timedOut = False


Voilà la structure de la réponse, sans rien deviner :
- **`notices`** : la liste des annonces demandées (ici 3, à cause de `limit=3`) ;
- **`totalNoticeCount`** : le nombre **total** d'annonces correspondant à ma recherche ;
- **`iterationNextToken`** : un jeton pour récupérer la suite (pagination) ;
- **`timedOut`** : si la requête a expiré.

C'est ici que je découvre **`totalNoticeCount`** : ce n'est pas un champ que je demande, c'est une
clé que l'API met dans sa réponse. C'est lui qui me donne le nombre de **lignes** (annonces).

## 5. Combien d'annonces existent ? (lignes, pas champs)

Je me fais une petite fonction pour lire `totalNoticeCount` selon une recherche.

In [7]:
def total_annonces(query, scope="ACTIVE"):
    body = {"query": query, "fields": ["publication-number"], "limit": 1, "scope": scope}
    return requests.post(URL, json=body).json().get("totalNoticeCount")

print("Annonces publiées depuis 1 jour  :", total_annonces("publication-date>=today(-1)"))
print("Annonces publiées depuis 7 jours :", total_annonces("publication-date>=today(-7)"))

Annonces publiées depuis 1 jour  : 7320
Annonces publiées depuis 7 jours : 21952


Plusieurs milliers par jour. Sur le **site** TED, sans filtre, j'obtiens ~6 millions. Pourquoi
l'écart ? Parce que le site montre **tout l'historique**, alors que je filtre sur les derniers jours.
Je le vérifie en enlevant le filtre de date (scope ALL).

In [8]:
def total_all(query):
    body = {"query": query, "fields": ["publication-number"], "limit": 1, "scope": "ALL"}
    return requests.post(URL, json=body).json().get("totalNoticeCount")

print("Tout l'historique :", total_all("publication-date>=today(-9000)"))
print("Depuis 1 an       :", total_all("publication-date>=today(-365)"))
print("Depuis 1 jour     :", total_all("publication-date>=today(-1)"))

Tout l'historique : 6937929
Depuis 1 an       : 900320
Depuis 1 jour     : 7321


Confirmé : tout l'historique donne ~6-7 millions, comme le site. L'API et le site sont donc
cohérents. **Conséquence :** pour ma veille, je filtrerai toujours par date récente + cyber + scope
ACTIVE, et je ferai tourner la collecte chaque jour (sinon le volume est ingérable).

## 6. Une fonction de recherche réutilisable

In [9]:
CHAMPS = ["publication-number", "notice-title", "buyer-name", "buyer-country",
          "publication-date", "deadline", "notice-type", "classification-cpv", "links"]

def chercher(query, limit=50, scope="ACTIVE"):
    body = {"query": query, "fields": CHAMPS, "limit": limit, "page": 1, "scope": scope}
    r = requests.post(URL, json=body, timeout=40)
    r.raise_for_status()
    return r.json()

## 7. Regarder une annonce en détail

Avant de filtrer, je regarde la structure brute d'une annonce (c'est là qu'on trouve les surprises).

In [10]:
data = chercher("publication-date>=today(-2)", limit=5)
print(json.dumps(data["notices"][0], indent=2, ensure_ascii=False)[:1000])

{
  "notice-type": "cn-standard",
  "publication-number": "413513-2026",
  "classification-cpv": [
    "60112000",
    "60112000"
  ],
  "buyer-name": {
    "deu": [
      "Zweckverband Verkehrsverbund Bremen/Niedersachsen (ZVBN)"
    ]
  },
  "publication-date": "2026-06-17+02:00",
  "links": {
    "xml": {
      "MUL": "https://ted.europa.eu/en/notice/413513-2026/xml"
    },
    "pdf": {
      "BUL": "https://ted.europa.eu/bg/notice/413513-2026/pdf",
      "SPA": "https://ted.europa.eu/es/notice/413513-2026/pdf",
      "CES": "https://ted.europa.eu/cs/notice/413513-2026/pdf",
      "DAN": "https://ted.europa.eu/da/notice/413513-2026/pdf",
      "DEU": "https://ted.europa.eu/de/notice/413513-2026/pdf",
      "EST": "https://ted.europa.eu/et/notice/413513-2026/pdf",
      "ELL": "https://ted.europa.eu/el/notice/413513-2026/pdf",
      "ENG": "https://ted.europa.eu/en/notice/413513-2026/pdf",
      "FRA": "https://ted.europa.eu/fr/notice/413513-2026/pdf",
      "GLE": "https://ted.europ

J'observe trois formats à gérer : le **titre** est un dictionnaire par langue, les **dates** ont
un fuseau (ex. `2026-06-18+02:00`), et `classification-cpv` est une **liste** de codes.

## 8. Isoler la cybersécurité : la nomenclature CPV officielle

Sur le site, à gauche, des filtres regroupent les annonces par grandes catégories (« Computer and
Related Services »...). En cliquant, j'ai vu qu'elles correspondent à des **codes CPV**. Mais le site
affiche aussi des **nombres** d'annonces par catégorie, et je ne veux PAS identifier mes codes à
partir de ces nombres : ils changent chaque jour (de nouvelles annonces arrivent), donc ce ne serait
pas reproductible.

La source **rigoureuse et stable**, c'est la **nomenclature CPV officielle** de l'UE : un fichier qui
associe chaque code à son libellé, et qui ne change pas dans le temps (version 2008, toujours en
vigueur). Je l'ai téléchargé une fois depuis TED/SIMAP et rangé dans mon projet. Je le charge.

In [11]:
import pandas as pd

# Fichier officiel CPV téléchargé depuis TED/SIMAP (adapte le chemin si besoin)
CHEMIN_CPV = "data/cpv_2008.xlsx"
cpv = pd.read_excel(CHEMIN_CPV)
print("Dimensions (codes, langues) :", cpv.shape)
print("Colonnes (langues) :", list(cpv.columns)[:12], "...")
cpv.head(3)

FileNotFoundError: [Errno 2] No such file or directory: 'data/cpv_2008.xlsx'

9454 codes, une colonne par langue (EN, FR, etc.). Je cherche les codes dont le **libellé anglais**
contient des mots liés à mon métier : audit, security, testing, penetration.

In [ ]:
mots = ["audit", "security", "testing", "penetration"]
masque_mot = cpv["EN"].str.lower().str.contains("|".join(mots), na=False)
print("Codes dont le libellé contient ces mots :", masque_mot.sum())
# aperçu rapide
for _, l in cpv[masque_mot].head(8).iterrows():
    print(f"  {str(l['CODE']).split('-')[0]:10} | {l['EN']}")

Attention : il y en a beaucoup (~85), mais avec du **bruit** : « Blood-testing reagents »,
« Convex security mirrors », « Cybercafé services »... Le mot « testing » ou « security » apparaît
dans des domaines qui n'ont rien à voir avec la cybersécurité. Je dois donc **trier**.

La cybersécurité informatique est surtout dans la **branche 72** (services informatiques) et quelques
codes **487** (logiciels de sécurité). Je combine donc : libellé pertinent **ET** code dans ces
branches.

In [ ]:
cpv["code8"] = cpv["CODE"].str.split("-").str[0]
masque_info = cpv["code8"].str.startswith("72") | cpv["code8"].str.startswith("487")
pertinents = cpv[masque_mot & masque_info]

print("Codes CPV retenus (1re recherche) :\n")
for _, l in pertinents.iterrows():
    print(f"  {l['code8']:10} | {l['EN']}")

Une quinzaine de codes (audit, essai, logiciels de sécurité...). Mais je me demande si je n'ai
pas **raté** des codes en ne cherchant que 4 mots. La cybersécurité, c'est aussi l'antivirus, le
pare-feu, le chiffrement, la sauvegarde, la reprise après sinistre... J'élargis donc ma recherche
de mots dans la nomenclature, toujours dans les branches informatique/logiciel.

In [ ]:
termes_larges = ["firewall", "intrusion", "encryption", "antivirus", "virus", "malware",
                 "network security", "information security", "cryptograph", "authentication",
                 "access control", "cyber", "data protection", "disaster recovery", "backup"]
masque_large = cpv["EN"].str.lower().str.contains("|".join(termes_larges), na=False)
nouveaux = cpv[masque_large & masque_info]

# ce que cette 2e recherche ajoute par rapport à la 1re
deja = set(pertinents["code8"])
ajouts = nouveaux[~nouveaux["code8"].isin(deja)]
print("Codes CPV spécifiques SUPPLÉMENTAIRES trouvés :\n")
for _, l in ajouts.iterrows():
    print(f"  {l['code8']:10} | {l['EN']}")

La recherche élargie révèle des codes que j'avais ratés : antivirus (`48760000`, `48761000`),
sauvegarde (`48710000`), reprise après sinistre (`72251000`), et leurs services de développement.
Ce sont bien des codes **spécifiques** à la sécurité, donc je les ajoute à ma liste.

Ma liste finale combine les deux recherches.

In [ ]:
CODES_CPV_CYBER = sorted(set(pertinents["code8"]) | set(nouveaux["code8"]))
print(f"Liste finale : {len(CODES_CPV_CYBER)} codes CPV spécifiques")
print(CODES_CPV_CYBER)

**Une tentation à éviter :** pourquoi ne pas ajouter aussi les codes **génériques** que je vois
parfois sur des annonces cyber, comme `72000000` (services informatiques) ? Parce qu'un code
générique ne signifie pas « cyber » : il couvre TOUT l'informatique. Je le vérifie en comptant
combien d'annonces actives il représente.

In [ ]:
# Combien d'annonces pour un code GÉNÉRIQUE vs nos codes SPÉCIFIQUES ?
print("72000000 (services informatiques, générique) :", total_annonces("classification-cpv=72000000"))
print("72810000 (audit informatique, spécifique)    :", total_annonces("classification-cpv=72810000"))

Le code générique `72000000` représente des **dizaines de milliers** d'annonces (tout
l'informatique : sites web, ERP, hébergement...), alors qu'un code spécifique comme `72810000` en
donne quelques centaines. Ajouter les codes génériques **noierait** mes vraies opportunités cyber
sous des milliers d'annonces hors sujet.

**Donc :** je garde uniquement les codes CPV **spécifiques**. Pour attraper les annonces cyber
classées sous un code générique (ex. un pentest rangé en `72000000`), je m'appuie sur les
**mots-clés** du titre, pas sur les codes génériques. Et le **scoring IA** écarte le bruit résiduel.
C'est la combinaison : CPV spécifiques + mots-clés + IA.

## 9. Les mots-clés : faut-il chercher dans d'autres langues ?

Les codes CPV captent les annonces bien classées. Pour les annonces cyber classées sous un code
**générique** (ex. un pentest rangé en `72000000`), je m'appuie sur des **mots-clés** dans le titre.
Mes mots-clés sont en anglais. Mais les titres TED existent dans 24 langues : est-ce que je rate les
annonces dont le titre n'est pas en anglais ? Je teste en comparant un mot anglais à ses équivalents
dans d'autres langues.

In [ ]:
def total(q):
    body = {"query": q + " AND notice-type=cn-standard", "fields": ["publication-number"],
            "limit": 1, "scope": "ACTIVE"}
    return requests.post(URL, json=body, timeout=60).json()["totalNoticeCount"]

print("ANGLAIS  'vulnerability'  :", total('notice-title ~ "vulnerability"'))
print("FRANÇAIS 'vulnérabilité'  :", total('notice-title ~ "vulnérabilité"'))
print("ALLEMAND 'Sicherheit'     :", total('notice-title ~ "Sicherheit"'))
print("POLONAIS 'bezpieczeństwa' :", total('notice-title ~ "bezpieczeństwa"'))

Conclusion : chaque langue a ses propres annonces (l'allemand et le polonais en ont des
milliers !), donc mes mots anglais en ratent beaucoup. **Mais attention** : « Sicherheit » (allemand)
ou « bezpieczeństwa » (polonais) veulent dire « sécurité » au sens **large** (sécurité physique,
incendie...), pas seulement cyber. Si j'ajoute des mots aussi génériques, je ramène énormément de
bruit. Je dois donc choisir des termes **précis** par langue (l'équivalent de « cybersécurité »,
« test d'intrusion »), pas « sécurité » tout court. Je teste quelques termes précis.

In [ ]:
for nom, q in {
    "FR cybersécurité":       'notice-title ~ "cybersécurité"',
    "DE Cybersicherheit":     'notice-title ~ "Cybersicherheit"',
    "DE Penetrationstest":    'notice-title ~ "Penetrationstest"',
    "PL cyberbezpieczeństwo": 'notice-title ~ "cyberbezpieczeństwo"',
    "ES ciberseguridad":      'notice-title ~ "ciberseguridad"',
}.items():
    print(f"  {nom:24} : {total(q)}")

Ces termes précis donnent des volumes raisonnables (quelques dizaines à ~150), sans l'explosion
de « Sicherheit ». Ils valent donc la peine d'être ajoutés. Je ne vise pas les 24 langues (ce serait
disproportionné, et les codes CPV couvrent déjà le principal indépendamment de la langue) : j'ajoute
les langues majeures (français, allemand, polonais, espagnol) avec des termes précis.

## 10. Ma requête de collecte cyber (CPV + mots-clés multilingues)

Je combine les codes CPV trouvés (signal fort) ET des mots-clés cyber en plusieurs langues, en me
limitant aux mises en concurrence (`notice-type=cn-standard`), c'est-à-dire des appels d'offres
ouverts (pas des avis d'attribution déjà clôturés).

In [ ]:
filtre_cpv = " OR ".join(f"classification-cpv={c}" for c in CODES_CPV_CYBER)
filtre_mots = ('notice-title ~ "cybersecurity" OR notice-title ~ "penetration" '
               'OR notice-title ~ "ISO 27001" OR notice-title ~ "security audit" '
               'OR notice-title ~ "vulnerability" OR notice-title ~ "pentest" '
               'OR notice-title ~ "cybersécurité" OR notice-title ~ "cyberbezpieczeństwo" '
               'OR notice-title ~ "ciberseguridad" OR notice-title ~ "Cybersicherheit" '
               'OR notice-title ~ "Penetrationstest"')

REQUETE_CYBER = (f"({filtre_cpv} OR {filtre_mots}) "
                 "AND notice-type=cn-standard SORT BY publication-date DESC")

data = chercher(REQUETE_CYBER, limit=100)
print("Annonces cyber (mises en concurrence) :", data["totalNoticeCount"])
print("Récupérées dans l'échantillon :", len(data["notices"]))

## 11. Contrôle qualité des données

Avant de coder le collecteur, je vérifie la qualité sur cet échantillon réel.

In [ ]:
notices = data["notices"]
n = len(notices)
print(f"Échantillon : {n} annonces\n")
print("a) Complétude :")
for champ in CHAMPS:
    presents = sum(1 for x in notices if x.get(champ) not in (None, "", [], {}))
    print(f"   {champ:20} : {presents}/{n}  ({100*presents//max(n,1)}%)")

In [ ]:
print("b) Doublons sur publication-number :")
nums = [x.get("publication-number") for x in notices]
print(f"   {len(nums)} numéros, {len(set(nums))} uniques -> {len(nums)-len(set(nums))} doublon(s)")

In [ ]:
print("c) Langues des titres :")
sans_anglais = sum(1 for x in notices
                   if isinstance(x.get("notice-title"), dict) and "eng" not in x["notice-title"])
print(f"   annonces sans titre anglais : {sans_anglais}/{n}")

In [ ]:
print("d) Date limite (deadline) :")
def premiere(d): return d[0] if isinstance(d, list) else d
avec_dl = [x for x in notices if x.get("deadline")]
print(f"   présente dans : {len(avec_dl)}/{n}")
if avec_dl:
    print("   exemple de format :", premiere(avec_dl[0]["deadline"]))
    auj = date.today().isoformat()
    passees = sum(1 for x in avec_dl if str(premiere(x["deadline"]))[:10] < auj)
    print(f"   déjà passées alors que scope=ACTIVE : {passees}/{len(avec_dl)}")

In [ ]:
print("e) Lien : je le reconstruis depuis le numéro")
num = notices[0]["publication-number"]
print("   ", f"https://ted.europa.eu/fr/notice/{num}")

### Ce que j'apprends

- Champs **fiables à 100 %** : `publication-number`, `notice-title`, `buyer-name`,
  `buyer-country`, `publication-date`, `classification-cpv`, `links`.
- **`publication-number` unique** (0 doublon) -> ma clé.
- **Titres toujours en anglais** -> je prends `eng`.
- **`deadline` souvent absente, et parfois déjà passée même en scope ACTIVE.** Je ne m'y fie donc
  pas seule : si elle est présente et dépassée, l'annonce est close ; sinon je me base sur `ACTIVE`.
  Format à normaliser : `2026-06-29T15:00:00+01:00` -> `2026-06-29`.
- **Liens** : structure complexe -> je reconstruis `https://ted.europa.eu/fr/notice/<num>`.

## 12. Les détails riches (prix, procédure, gagnant) sont-ils dans l'API ?

En cliquant sur une annonce du site, j'ai vu plein de détails : valeur estimée (le prix), type de
procédure, durée, gagnant, email de l'acheteur. Est-ce que l'API a ces champs ? Je cherche dans la
liste.

In [ ]:
for label, ms in {"valeur/prix":["estimated-value","value-cur"], "procédure":["procedure-type"],
                  "durée":["contract-duration"], "gagnant":["winner"],
                  "description":["description-lot","description-proc"]}.items():
    trouves = []
    for m in ms:
        trouves += [c for c in champs if m in c.lower() and not c.startswith("BT-")]
    print(f"{label:14} -> {trouves[:4]}")

Oui, l'API contient ces détails. Donc **pas besoin de scraper la page de détail** : tout est
dans l'API, il suffit d'ajouter les champs voulus à ma requête.

**Stratégie (la même que pour la BCEAO) :** je ne récupère pas tout pour tout le monde.
- À la collecte : champs **essentiels** (titre, organisme, pays, date, lien, CPV) pour toutes les
  annonces ouvertes.
- Après le scoring : champs **riches** (prix, procédure, durée, description, documents) seulement
  pour les annonces jugées **pertinentes**, en enrichissant la requête API.

Remarque : un avis avec une section « Results / Winner » est un **avis d'attribution** (marché déjà
gagné), pas une opportunité. Le filtre `notice-type=cn-standard` les écarte.

## 13. Décisions de conception pour `collecter_ted()`

Pour alimenter la même base que la BCEAO, je produis le même format :

| Champ de ma base | Source TED |
|---|---|
| `cle_unique` | `ted::<publication-number>` |
| `intitule` | `notice-title["eng"]` |
| `source` | `"TED"` |
| organisme / pays | `buyer-name` / `buyer-country` |
| `date_publication` | `publication-date` normalisée |
| `delai_soumission` | `deadline` si présent, normalisé, sinon `null` |
| `lien` | `https://ted.europa.eu/fr/notice/<numéro>` |
| `statut_source` | `"en cours"` (ACTIVE), ou `"clôturé"` si date limite dépassée |

Le prix et les détails riches seront ajoutés **plus tard**, pour les annonces pertinentes.

## 14. Exploration des détails des annonces pertinentes

Maintenant que j'ai des annonces pertinentes, je veux récupérer leurs **détails riches** (prix,
description, procédure, durée) pour les afficher dans le digest. Mais avant de décider quoi stocker,
je vérifie ce qui existe vraiment dans l'API et ce qui est réellement rempli. Je ne stocke pas un
champ au hasard : je teste d'abord.

In [ ]:
# D'abord, quels champs de détail existent ? Je réutilise l'astuce du champ bidon.
r = requests.post(URL, json={"query": 'publication-date>=today(-2)', "fields": ["xxx"], "limit": 1})
tous = [c.strip() for c in r.json()["message"].split("supported values are:")[1].rstrip(")").split(",")]

for label, motcle in {"prix": "estimated-value", "procédure": "procedure-type",
                      "durée": "contract-duration", "date limite fiable": "deadline-receipt-tender",
                      "description": "description-proc"}.items():
    trouves = [c for c in tous if motcle in c.lower() and not c.startswith("BT-")][:3]
    print(f"{label:20} -> {trouves}")

Les champs existent. Mais **existent** ne veut pas dire **remplis**. Je teste le taux de
remplissage sur un échantillon réel d'annonces cyber, pour savoir lesquels sont fiables.

In [ ]:
champs_detail = ["publication-number", "description-proc", "procedure-type",
                 "deadline-receipt-tender-date-lot", "estimated-value-lot"]
data = chercher(REQUETE_CYBER, limit=30)
notices = data["notices"]
# je redemande ces annonces avec les champs de détail
nums = [n["publication-number"] for n in notices][:20]
q = " OR ".join(f'publication-number="{x}"' for x in nums)
d2 = requests.post(URL, json={"query": q, "fields": champs_detail, "limit": 20, "scope": "ALL"}).json()

n = len(d2["notices"])
print(f"Sur {n} annonces, remplissage :")
for c in champs_detail[1:]:
    presents = sum(1 for x in d2["notices"] if x.get(c) not in (None, "", [], {}))
    print(f"  {c:34} : {presents}/{n}")

J'observe que la **description** et la **procédure** sont presque toujours présentes, que la
**date limite** (`deadline-receipt-tender-date-lot`) est bien mieux remplie que le champ `deadline`
que j'utilisais (rappel : 13 % seulement), mais que le **prix** (`estimated-value-lot`) est souvent
absent.

**Attention au raisonnement :** le prix absent sur cet échantillon ne veut PAS dire qu'il est
inutile. D'autres annonces (aujourd'hui ou demain) l'auront. Donc je ne dois pas **exclure** un
champ parce qu'il est parfois vide : je **récupère tout** ce qui est utile, et je **stocke ce qui
est présent** pour chaque annonce (les champs absents restent `null`).

### La langue des descriptions : une surprise

Pour le titre, j'avais vu qu'il est toujours traduit en anglais. Je vérifie si c'est pareil pour la
description.

In [ ]:
def langue_desc(champ):
    return list(champ.keys()) if isinstance(champ, dict) and champ else []

from collections import Counter
langues = Counter()
for x in d2["notices"]:
    for lg in langue_desc(x.get("description-proc")):
        langues[lg] += 1
print("langues des descriptions :", dict(langues.most_common()))
print("descriptions en anglais  :", langues.get("eng", 0), "/", n)

**Découverte importante :** contrairement au titre, la description n'est **PAS traduite** : elle
n'existe que dans la **langue d'origine** du pays (polonais, allemand, roumain...). Si je choisissais
"anglais", je n'aurais quasiment aucune description.

**Conséquence sur ma conception :**
- je récupère la description dans la langue **disponible** : français d'abord (pour l'utilisateur
  francophone), sinon anglais, sinon la langue d'origine ;
- je note un **angle mort** : les descriptions étrangères devront être **traduites** pour
  l'utilisateur (amélioration future, via une IA de traduction).

### Décision finale pour l'enrichissement

Pour les annonces **pertinentes** seulement (pas toutes, pour économiser les appels), je récupère et
je stocke : la **description** (langue dispo), la **procédure**, une **date limite fiable**, et le
**prix** quand il est présent. Cela alimente la fonction `enrichir_pertinentes()` du script.

### Mise en pratique : récupérer les détails d'une annonce pertinente

Je montre maintenant, sur de vraies annonces, comment je récupère et prépare leurs détails. C'est la
logique qui deviendra `enrichir_pertinentes()` dans mon script. Je pars de quelques numéros
d'annonces (en pratique, les `publication-number` des annonces jugées pertinentes), et je redemande
à l'API leurs champs de détail.

In [ ]:
echantillon = chercher(REQUETE_CYBER, limit=5)["notices"]
nums_pertinents = [n["publication-number"] for n in echantillon]
print("Numéros à enrichir :", nums_pertinents)

q = " OR ".join(f'publication-number="{x}"' for x in nums_pertinents)
champs_detail = ["publication-number", "description-proc", "procedure-type",
                 "deadline-receipt-tender-date-lot", "estimated-value-lot", "estimated-value-cur-lot"]
detail = requests.post(URL, json={"query": q, "fields": champs_detail,
                                  "limit": 5, "scope": "ALL"}).json()["notices"]
print("Détails récupérés pour", len(detail), "annonces")

J'extrais proprement chaque information. Le point délicat est la **description** : elle est dans
la langue d'origine, donc je prends le français d'abord, sinon l'anglais, sinon n'importe quelle
langue disponible.

In [ ]:
def extraire_description(champ):
    """Description dans la langue dispo : français, sinon anglais, sinon n'importe laquelle."""
    if isinstance(champ, dict) and champ:
        for lg in ("fra", "eng"):
            if champ.get(lg):
                v = champ[lg]
                return (v[0] if isinstance(v, list) else v), lg
        for lg, v in champ.items():
            return (v[0] if isinstance(v, list) else v), lg
    return None, None

def premier(v):
    """Beaucoup de champs TED sont des listes : je prends le 1er élément."""
    return v[0] if isinstance(v, list) and v else (None if isinstance(v, list) else v)

for n in detail:
    desc, langue = extraire_description(n.get("description-proc"))
    print(f"\nAnnonce {n['publication-number']}")
    print(f"  procédure   : {premier(n.get('procedure-type'))}")
    print(f"  date limite : {str(premier(n.get('deadline-receipt-tender-date-lot')))[:10]}")
    print(f"  prix        : {premier(n.get('estimated-value-lot'))} {premier(n.get('estimated-value-cur-lot'))}")
    print(f"  description [{langue}] : {(desc or '(aucune)')[:90]}")

Je vérifie le point important : quand un champ est **absent** (souvent le prix), mon extraction
renvoie `None` **sans planter**. C'est ce qui permet de stocker ce qui existe et de laisser le reste
vide.

In [ ]:
sans_prix = [n for n in detail if premier(n.get("estimated-value-lot")) is None]
print(f"{len(sans_prix)}/{len(detail)} annonces sans prix -> prix=None, sans erreur")
avec_desc = [n for n in detail if extraire_description(n.get("description-proc"))[0]]
print(f"{len(avec_desc)}/{len(detail)} annonces avec une description (quelle que soit la langue)")

**Conclusion :** cette logique (récupérer les détails par lot, extraire la description dans la
langue dispo, gérer les champs absents) est exactement ce que fait `enrichir_pertinentes()` dans le
script. La seule différence : le script ajoute l'**écriture dans MongoDB** (mettre à jour chaque
annonce pertinente avec ces champs, et la marquer `enrichi: true` pour ne pas refaire le travail).
L'écriture en base n'a pas sa place dans un notebook d'exploration.

## 15. Prochaines étapes

1. Écrire `collecter_ted()` avec les champs **essentiels**, testée sur de vraies données. (fait)
2. L'intégrer dans `collecte_et_scoring.py`, à côté de `collecter_bceao()`. (fait)
3. Vérifier que le scoring repère les vraies annonces cyber. (fait, prompt affiné)
4. Enrichir les pertinentes avec les détails (description, procédure, date limite, prix). (fait)
5. Plus tard : traduire les descriptions étrangères pour l'utilisateur francophone.